# INX Future Inc — Employee Performance Analysis
## Notebook 4: Prediction Pipeline
**Project Code:** 10281 | **IABAC Certified Data Scientist Project**

This notebook demonstrates how to use the trained Random Forest model to predict employee performance ratings for new candidates. This model will be used during the hiring process.

---
## 1. Load Model & Assets

In [5]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

with open('../../data/processed/best_model_rf.pkl', 'rb') as f:
    model = pickle.load(f)

with open('../../data/processed/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('../../data/processed/feature_columns.pkl', 'rb') as f:
    feature_columns = pickle.load(f)

print('Model, scaler, and feature columns loaded successfully.')
print(f'Model expects {len(feature_columns)} features.')

Model, scaler, and feature columns loaded successfully.
Model expects 56 features.


---
## 2. Define Prediction Function

In [6]:
def preprocess_employee(employee_dict):
    """
    Preprocess a raw employee record dict into model-ready feature vector.
    Applies the same encoding and scaling as training.
    """
    df_input = pd.DataFrame([employee_dict])

    # Binary encode
    df_input['Gender'] = 1 if df_input['Gender'].iloc[0] == 'Male' else 0
    df_input['OverTime'] = 1 if df_input['OverTime'].iloc[0] == 'Yes' else 0
    df_input['Attrition'] = 1 if df_input['Attrition'].iloc[0] == 'Yes' else 0

    # Ordinal encode travel
    travel_map = {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
    df_input['BusinessTravelFrequency'] = travel_map.get(df_input['BusinessTravelFrequency'].iloc[0], 0)

    # One-hot encode
    nominal_cols = ['EducationBackground', 'MaritalStatus', 'EmpDepartment', 'EmpJobRole']
    df_input = pd.get_dummies(df_input, columns=nominal_cols)

    # Align with training columns
    for col in feature_columns:
        if col not in df_input.columns:
            df_input[col] = 0
    df_input = df_input[feature_columns]

    # Scale numerical features
    num_cols = ['Age', 'DistanceFromHome', 'EmpEducationLevel', 'EmpEnvironmentSatisfaction',
                'EmpHourlyRate', 'EmpJobInvolvement', 'EmpJobLevel', 'EmpJobSatisfaction',
                'NumCompaniesWorked', 'EmpLastSalaryHikePercent', 'EmpRelationshipSatisfaction',
                'TotalWorkExperienceInYears', 'TrainingTimesLastYear', 'EmpWorkLifeBalance',
                'ExperienceYearsAtThisCompany', 'ExperienceYearsInCurrentRole',
                'YearsSinceLastPromotion', 'YearsWithCurrManager']
    existing_num = [c for c in num_cols if c in df_input.columns]
    df_input[existing_num] = scaler.transform(df_input[existing_num])

    return df_input


def predict_performance(employee_dict):
    """
    Takes raw employee data dict and returns predicted performance rating with confidence.
    """
    rating_map = {2: 'Low', 3: 'Good', 4: 'Excellent'}
    processed = preprocess_employee(employee_dict)
    prediction = model.predict(processed)[0]
    probabilities = model.predict_proba(processed)[0]
    classes = model.classes_
    prob_dict = {rating_map[c]: f'{p*100:.1f}%' for c, p in zip(classes, probabilities)}
    return prediction, rating_map[prediction], prob_dict


print('Prediction function defined.')

Prediction function defined.


---
## 3. Sample Predictions — New Candidates

Below are three sample employee profiles (as would be collected during hiring) passed through the prediction pipeline.

In [7]:
# Candidate 1 — Likely Excellent performer
candidate_1 = {
    'Age': 35, 'Gender': 'Male', 'EducationBackground': 'Life Sciences',
    'MaritalStatus': 'Married', 'EmpDepartment': 'Data Science',
    'EmpJobRole': 'Data Scientist', 'BusinessTravelFrequency': 'Travel_Rarely',
    'DistanceFromHome': 5, 'EmpEducationLevel': 4, 'EmpEnvironmentSatisfaction': 4,
    'EmpHourlyRate': 80, 'EmpJobInvolvement': 4, 'EmpJobLevel': 3,
    'EmpJobSatisfaction': 4, 'NumCompaniesWorked': 2, 'OverTime': 'No',
    'EmpLastSalaryHikePercent': 20, 'EmpRelationshipSatisfaction': 4,
    'TotalWorkExperienceInYears': 12, 'TrainingTimesLastYear': 4,
    'EmpWorkLifeBalance': 3, 'ExperienceYearsAtThisCompany': 6,
    'ExperienceYearsInCurrentRole': 4, 'YearsSinceLastPromotion': 1,
    'YearsWithCurrManager': 3, 'Attrition': 'No'
}

# Candidate 2 — Likely Low performer
candidate_2 = {
    'Age': 28, 'Gender': 'Female', 'EducationBackground': 'Human Resources',
    'MaritalStatus': 'Single', 'EmpDepartment': 'Human Resources',
    'EmpJobRole': 'Human Resources', 'BusinessTravelFrequency': 'Travel_Frequently',
    'DistanceFromHome': 25, 'EmpEducationLevel': 2, 'EmpEnvironmentSatisfaction': 1,
    'EmpHourlyRate': 40, 'EmpJobInvolvement': 1, 'EmpJobLevel': 1,
    'EmpJobSatisfaction': 1, 'NumCompaniesWorked': 6, 'OverTime': 'Yes',
    'EmpLastSalaryHikePercent': 11, 'EmpRelationshipSatisfaction': 1,
    'TotalWorkExperienceInYears': 4, 'TrainingTimesLastYear': 1,
    'EmpWorkLifeBalance': 1, 'ExperienceYearsAtThisCompany': 1,
    'ExperienceYearsInCurrentRole': 1, 'YearsSinceLastPromotion': 4,
    'YearsWithCurrManager': 1, 'Attrition': 'Yes'
}

# Candidate 3 — Average performer
candidate_3 = {
    'Age': 42, 'Gender': 'Male', 'EducationBackground': 'Technical Degree',
    'MaritalStatus': 'Divorced', 'EmpDepartment': 'Development',
    'EmpJobRole': 'Developer', 'BusinessTravelFrequency': 'Travel_Rarely',
    'DistanceFromHome': 10, 'EmpEducationLevel': 3, 'EmpEnvironmentSatisfaction': 3,
    'EmpHourlyRate': 65, 'EmpJobInvolvement': 3, 'EmpJobLevel': 2,
    'EmpJobSatisfaction': 3, 'NumCompaniesWorked': 3, 'OverTime': 'No',
    'EmpLastSalaryHikePercent': 14, 'EmpRelationshipSatisfaction': 3,
    'TotalWorkExperienceInYears': 15, 'TrainingTimesLastYear': 3,
    'EmpWorkLifeBalance': 3, 'ExperienceYearsAtThisCompany': 8,
    'ExperienceYearsInCurrentRole': 4, 'YearsSinceLastPromotion': 2,
    'YearsWithCurrManager': 4, 'Attrition': 'No'
}

for i, candidate in enumerate([candidate_1, candidate_2, candidate_3], 1):
    rating_num, rating_label, probs = predict_performance(candidate)
    print(f'Candidate {i}:')
    print(f'  Department : {candidate["EmpDepartment"]}')
    print(f'  Role       : {candidate["EmpJobRole"]}')
    print(f'  Prediction : Rating {rating_num} — {rating_label}')
    print(f'  Confidence : {probs}')
    print()

Candidate 1:
  Department : Data Science
  Role       : Data Scientist
  Prediction : Rating 4 — Excellent
  Confidence : {'Low': '7.0%', 'Good': '24.5%', 'Excellent': '68.5%'}

Candidate 2:
  Department : Human Resources
  Role       : Human Resources
  Prediction : Rating 3 — Good
  Confidence : {'Low': '29.5%', 'Good': '64.0%', 'Excellent': '6.5%'}

Candidate 3:
  Department : Development
  Role       : Developer
  Prediction : Rating 3 — Good
  Confidence : {'Low': '3.0%', 'Good': '88.5%', 'Excellent': '8.5%'}



---
## 4. Batch Prediction on New Data

The model can accept a CSV or Excel file of new candidates and output predictions in bulk.

In [8]:
# Demonstrate batch prediction with sample records
candidates = [candidate_1, candidate_2, candidate_3]
results = []

for i, c in enumerate(candidates, 1):
    rating_num, rating_label, probs = predict_performance(c)
    results.append({
        'Candidate': i,
        'Department': c['EmpDepartment'],
        'Role': c['EmpJobRole'],
        'Predicted Rating': rating_num,
        'Rating Label': rating_label,
        'Confidence Low': probs.get('Low', '0%'),
        'Confidence Good': probs.get('Good', '0%'),
        'Confidence Excellent': probs.get('Excellent', '0%')
    })

results_df = pd.DataFrame(results)
print('Batch Prediction Results:')
results_df

Batch Prediction Results:


,Candidate,Department,Role,Predicted Rating,Rating Label,Confidence Low,Confidence Good,Confidence Excellent
0,1,Data Science,Data Scientist,4,Excellent,7.0%,24.5%,68.5%
1,2,Human Resources,Human Resources,3,Good,29.5%,64.0%,6.5%
2,3,Development,Developer,3,Good,3.0%,88.5%,8.5%


---
## 5. How to Use This Model in Hiring

1. Collect candidate data matching the 27 input features during the hiring process
2. Pass the data dictionary (or CSV row) into `predict_performance()`
3. Candidates predicted as **Excellent (4)** are prioritized
4. Candidates predicted as **Low (2)** are flagged for further review
5. The confidence scores help HR quantify certainty before making decisions

> **Important:** This model is a decision-support tool. Final hiring decisions must involve human judgment, especially when confidence scores are low (e.g., < 60%).